# Data-driven styling: an accessibility audit

Turn a small GeoJSON dataset into an explorable accessibility map. Rather than assigning a Python color to every feature, this notebook lets MapLibre style the data in the browser with expressions.

The audit records below are **illustrative synthetic data**. They are embedded in the notebook so the example remains reproducible and works offline after MapLibre's web assets are cached.

In [ ]:
from maplibreum import Legend, Map

segments = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {"type": "LineString", "coordinates": [[132.4488, 34.3930], [132.4525, 34.3927]]},
            "properties": {"name": "Peace Boulevard west", "status": "good", "surface": "paved", "score": 92},
        },
        {
            "type": "Feature",
            "geometry": {"type": "LineString", "coordinates": [[132.4525, 34.3927], [132.4565, 34.3928]]},
            "properties": {"name": "Peace Boulevard east", "status": "attention", "surface": "paving_stones", "score": 64},
        },
        {
            "type": "Feature",
            "geometry": {"type": "LineString", "coordinates": [[132.4523, 34.3927], [132.4524, 34.3967], [132.4554, 34.3991]]},
            "properties": {"name": "Riverside connection", "status": "good", "surface": "asphalt", "score": 84},
        },
        {
            "type": "Feature",
            "geometry": {"type": "LineString", "coordinates": [[132.4554, 34.3991], [132.4594, 34.4028]]},
            "properties": {"name": "Castle approach", "status": "barrier", "surface": "gravel", "score": 28},
        },
        {
            "type": "Feature",
            "geometry": {"type": "LineString", "coordinates": [[132.4594, 34.4028], [132.4632, 34.4012], [132.4674, 34.3993]]},
            "properties": {"name": "Garden connector", "status": "attention", "surface": "compacted", "score": 57},
        },
    ],
}

print(f"Prepared {len(segments['features'])} audited segments")

## Style properties, not Python objects

The `match` expression maps categories to colors. The `interpolate` expression continuously maps the audit score to line width. These expressions stay live in the browser and scale naturally to vector tiles and large GeoJSON sources.

In [ ]:
audit_map = Map(
    center=[132.458, 34.397],
    zoom=14,
    title="Illustrative pedestrian accessibility audit",
    height="580px",
)
audit_map.add_source("audit", {"type": "geojson", "data": segments})
audit_map.add_line_layer(
    "audited-walkways",
    source="audit",
    paint={
        "line-color": [
            "match", ["get", "status"],
            "good", "#2a9d8f",
            "attention", "#f4a261",
            "barrier", "#e63946",
            "#6c757d",
        ],
        "line-width": ["interpolate", ["linear"], ["get", "score"], 0, 3, 100, 9],
        "line-opacity": 0.9,
    },
)

audit_map.add_feature_popup(
    "audited-walkways",
    fields=["name", "status", "surface", "score"],
    aliases=["Segment", "Assessment", "Surface", "Score"],
    title="Accessibility audit",
)
audit_map.add_tooltip(layer_id="audited-walkways", prop="name")
Legend([
    ("Good", "#2a9d8f"),
    ("Needs attention", "#f4a261"),
    ("Barrier", "#e63946"),
]).add_to(audit_map)
audit_map.add_control("navigation", options={"showZoom": True})
audit_map.add_control("fullscreen")
audit_map.fit_bounds([[132.447, 34.390], [132.469, 34.405]], padding=55)

audit_map

## Why this pattern matters

The same layer definition works when `audit` changes from embedded GeoJSON to a remote GeoJSON URL, a vector-tile source, or a PMTiles archive. Python prepares the map specification; MapLibre performs the interactive rendering.

Try changing one feature's `status` or `score` and rerun the map cell. Next, we will aggregate values into a thematic choropleth.